<a href="https://colab.research.google.com/github/jyotshna068/Traffic_Demand_Ensemble/blob/main/Gridlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')


In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (77299, 11)
Test shape: (41778, 10)


In [ ]:
def extract_time_features(df):
    # Parse timestamp like "0:0", "2:15", "18:45"
    df['hour'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(str(x).split(':')[1]))
    df['time_in_minutes'] = df['hour'] * 60 + df['minute']

    # Cyclic encoding for time
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Day features
    df['day_mod7'] = df['day'] % 7  # day of week proxy
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 7)

    return df

train = extract_time_features(train)
test = extract_time_features(test)

In [ ]:
geo_stats = train.groupby('geohash')['demand'].agg(['mean', 'median', 'std', 'count']).reset_index()
geo_stats.columns = ['geohash', 'geo_demand_mean', 'geo_demand_median', 'geo_demand_std', 'geo_count']
geo_stats['geo_demand_std'] = geo_stats['geo_demand_std'].fillna(0)

train = train.merge(geo_stats, on='geohash', how='left')
test = test.merge(geo_stats, on='geohash', how='left')

In [ ]:
# Fill with global mean for unseen geohashes
global_mean = train['demand'].mean()
for col in ['geo_demand_mean', 'geo_demand_median', 'geo_demand_std']:
    train[col] = train[col].fillna(global_mean)
    test[col] = test[col].fillna(global_mean)
train['geo_count'] = train['geo_count'].fillna(0)
test['geo_count'] = test['geo_count'].fillna(0)

# Time-geohash interaction: avg demand per geohash per hour
geo_hour_stats = train.groupby(['geohash', 'hour'])['demand'].mean().reset_index()
geo_hour_stats.columns = ['geohash', 'hour', 'geo_hour_mean']
train = train.merge(geo_hour_stats, on=['geohash', 'hour'], how='left')
test = test.merge(geo_hour_stats, on=['geohash', 'hour'], how='left')
test['geo_hour_mean'] = test['geo_hour_mean'].fillna(test['geo_demand_mean'])
train['geo_hour_mean'] = train['geo_hour_mean'].fillna(train['geo_demand_mean'])

In [ ]:
train['geo_prefix3'] = train['geohash'].str[:3]
train['geo_prefix4'] = train['geohash'].str[:4]
train['geo_prefix5'] = train['geohash'].str[:5]
test['geo_prefix3'] = test['geohash'].str[:3]
test['geo_prefix4'] = test['geohash'].str[:4]
test['geo_prefix5'] = test['geohash'].str[:5]

# Categorical encoding
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_prefix3', 'geo_prefix4', 'geo_prefix5']
for col in cat_cols:
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

le = LabelEncoder()
for col in cat_cols:
    combined = pd.concat([train[col], test[col]], axis=0)
    le.fit(combined)
    train[col + '_enc'] = le.transform(train[col])
    test[col + '_enc'] = le.transform(test[col])

# Fill Temperature with median per Weather type
temp_median = train.groupby('Weather')['Temperature'].median()
train['Temperature'] = train.apply(
    lambda row: temp_median.get(row['Weather'], train['Temperature'].median()) if pd.isna(row['Temperature']) else row['Temperature'], axis=1)
test['Temperature'] = test.apply(
    lambda row: temp_median.get(row['Weather'], train['Temperature'].median()) if pd.isna(row['Temperature']) else row['Temperature'], axis=1)

In [ ]:
print("Missing values in train DataFrame:")
print(train.isnull().sum())



Missing values in train DataFrame:
Index                0
geohash              0
day                  0
timestamp            0
demand               0
RoadType             0
NumberofLanes        0
LargeVehicles        0
Landmarks            0
Temperature          0
Weather              0
hour                 0
minute               0
time_in_minutes      0
hour_sin             0
hour_cos             0
day_mod7             0
day_sin              0
day_cos              0
geo_demand_mean      0
geo_demand_median    0
geo_demand_std       0
geo_count            0
geo_hour_mean        0
geo_prefix3          0
geo_prefix4          0
geo_prefix5          0
RoadType_enc         0
LargeVehicles_enc    0
Landmarks_enc        0
Weather_enc          0
geo_prefix3_enc      0
geo_prefix4_enc      0
geo_prefix5_enc      0
dtype: int64


In [ ]:
# --- TIME-BASED TRAIN/VALIDATION SPLIT ---

# Convert timestamp string (e.g., "2:15") to minutes from midnight
def timestamp_to_minutes(df):
    hours_minutes = df['timestamp'].str.split(':', expand=True).astype(int)
    return hours_minutes[0] * 60 + hours_minutes[1]

train['minutes'] = timestamp_to_minutes(train)

# Define validation condition:
# Day 49 records that are at or after 1:30 AM (90 minutes from midnight)
is_val = (train['day'] == 49) & (train['minutes'] >= 90)

# Split into train and validation sets chronologically
train_split = train[~is_val].copy()
val_split = train[is_val].copy()

print(f"Training set size: {len(train_split)} rows")
print(f"Validation set size: {len(val_split)} rows (Timestamps: {val_split['timestamp'].unique()})")

Training set size: 74602 rows
Validation set size: 2697 rows (Timestamps: ['1:30' '1:45' '2:0'])


In [ ]:
y_train = train_split['demand']
X_train = train_split.drop(columns=['demand'])
X_test = test.copy()
X_val = val_split.drop(columns=['demand'])
y_val = val_split['demand']

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostRegressor, Pool

In [ ]:
from sklearn.metrics import r2_score,root_mean_squared_error, mean_absolute_error

In [ ]:
# Define features and explicitly track missing values for categorical strings
categorical_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
features = ['minutes', 'NumberofLanes', 'Temperature', 'geo_demand_mean', 'geo_demand_median', 'geo_demand_std'] + categorical_cols

# Fill string missing values so CatBoost can properly interpret them as a distinct category
for col in categorical_cols:
    train_split[col] = train_split[col].astype(str).fillna('missing')
    val_split[col] = val_split[col].astype(str).fillna('missing')

X_train, y_train = train_split[features], train_split['demand']
X_val, y_val = val_split[features], val_split['demand']

# Build CatBoost specialized Pool objects
train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
val_pool = Pool(X_val, y_val, cat_features=categorical_cols)

# --- 5. INITIALIZE AND VALIDATE CATBOOST ---
print("🤖 Training CatBoost model on training split...")
model_val = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=100  # Prints evaluation every 100 iterations
)

# Train using early stopping on the validation set to prevent overfitting
model_val.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

# Evaluate performance
val_preds = np.clip(model_val.predict(val_pool), 0, None)
val_rmse = root_mean_squared_error(y_val, val_preds)
val_mae = mean_absolute_error(y_val, val_preds)

print("\n📊 === CATBOOST VALIDATION PERFORMANCE ===")
print(f"Validation RMSE: {val_rmse:.5f}")
print(f"Validation MAE:  {val_mae:.5f}")
print("==========================================\n")

🤖 Training CatBoost model on training split...
0:	learn: 0.1360734	test: 0.1378516	best: 0.1378516 (0)	total: 156ms	remaining: 2m 36s
100:	learn: 0.0399850	test: 0.0516475	best: 0.0516475 (100)	total: 10.6s	remaining: 1m 33s
200:	learn: 0.0373652	test: 0.0499857	best: 0.0499857 (200)	total: 20.6s	remaining: 1m 21s
300:	learn: 0.0357614	test: 0.0495071	best: 0.0494754 (295)	total: 27s	remaining: 1m 2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.04947537754
bestIteration = 295

Shrink model to first 296 iterations.

📊 === CATBOOST VALIDATION PERFORMANCE ===
Validation RMSE: 0.04948
Validation MAE:  0.02943



In [ ]:
val_r2 = r2_score(y_val, val_preds)

In [ ]:
print(val_r2)

0.8815626286123862


In [ ]:
from lightgbm.callback import early_stopping

lgbm_features = [
    'minutes',
    'NumberofLanes',
    'Temperature',
    'geo_demand_mean',
    'geo_demand_median',
    'geo_demand_std',
    'RoadType_enc',
    'LargeVehicles_enc',
    'Landmarks_enc',
    'Weather_enc',
    'hour_sin',
    'hour_cos',
    'day_mod7',
    'day_sin',
    'day_cos',
    'time_in_minutes',
    'geo_hour_mean',
    'geo_prefix3_enc',
    'geo_prefix4_enc',
    'geo_prefix5_enc'
]
lgbm_categorical_features = [
    'RoadType_enc',
    'LargeVehicles_enc',
    'Landmarks_enc',
    'Weather_enc',
    'day_mod7',
    'geo_prefix3_enc',
    'geo_prefix4_enc',
    'geo_prefix5_enc'
]

X_train_lgbm = train_split[lgbm_features]
y_train_lgbm = train_split['demand']
X_val_lgbm = val_split[lgbm_features]
y_val_lgbm = val_split['demand']

print("🤖 Training LightGBM model on training split...")
model_lgbm = LGBMRegressor(
    objective='regression_l1', # MAE as objective
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1
)

# Train using early stopping on the validation set
model_lgbm.fit(X_train_lgbm, y_train_lgbm,
            eval_set=[(X_val_lgbm, y_val_lgbm)],
            eval_metric='mae',
            callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
            categorical_feature=lgbm_categorical_features
)

# Evaluate performance
val_preds_lgbm = np.clip(model_lgbm.predict(X_val_lgbm), 0, None)
val_rmse_lgbm = root_mean_squared_error(y_val_lgbm, val_preds_lgbm)
val_mae_lgbm = mean_absolute_error(y_val_lgbm, val_preds_lgbm)
val_r2_lgbm = r2_score(y_val_lgbm, val_preds_lgbm)

print("\n📊 === LIGHTGBM VALIDATION PERFORMANCE ===")
print(f"Validation RMSE: {val_rmse_lgbm:.5f}")
print(f"Validation MAE:  {val_mae_lgbm:.5f}")
print(f"Validation R2:   {val_r2_lgbm:.5f}")
print("==========================================\n")

🤖 Training LightGBM model on training split...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1601
[LightGBM] [Info] Number of data points in the train set: 74602, number of used features: 19
[LightGBM] [Info] Start training from score 0.047439

📊 === LIGHTGBM VALIDATION PERFORMANCE ===
Validation RMSE: 0.03006
Validation MAE:  0.01933
Validation R2:   0.95628



In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the CatBoost Regressor with base parameters
cb_model = CatBoostRegressor(
    random_seed=42,
    verbose=False, # Suppress verbose output during grid search
    loss_function='RMSE',
    eval_metric='RMSE'
)

# Define the parameter grid to search
param_grid = {
    'iterations': [200, 300, 400], # Explore a range around the early stopping point
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=cb_model,
    param_grid=param_grid,
    cv=3, # Using 3-fold cross-validation for a balance of speed and robustness
    scoring='neg_root_mean_squared_error', # Using negative RMSE, as GridSearchCV maximizes score
    n_jobs=-1, # Use all available CPU cores
    verbose=1 # More verbose output for the grid search process
)

print("🤖 Starting CatBoost hyperparameter tuning (this may take some time)...")

# Fit GridSearchCV on the training data, specifying categorical features
grid_search.fit(X_train, y_train, cat_features=categorical_cols)

print("\n✅ CatBoost tuning complete!")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best RMSE score (on CV): {-grid_search.best_score_:.5f}") # Convert back to positive RMSE

# Re-evaluate the best model found by GridSearchCV on the original validation set
best_cb_model = grid_search.best_estimator_
val_preds_tuned = np.clip(best_cb_model.predict(X_val), 0, None)
val_rmse_tuned = root_mean_squared_error(y_val, val_preds_tuned)
val_mae_tuned = mean_absolute_error(y_val, val_preds_tuned)
val_r2_tuned = r2_score(y_val, val_preds_tuned)

print("\n📊 === TUNED CATBOOST VALIDATION PERFORMANCE ===")
print(f"Validation RMSE: {val_rmse_tuned:.5f}")
print(f"Validation MAE:  {val_mae_tuned:.5f}")
print(f"Validation R2:   {val_r2_tuned:.5f}")
print("=================================================\n")

🤖 Starting CatBoost hyperparameter tuning (this may take some time)...
Fitting 3 folds for each of 27 candidates, totalling 81 fits


KeyboardInterrupt: 

In [ ]:
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Reusing features and data split from LightGBM for consistency
X_train_xgb = X_train_lgbm.copy()
y_train_xgb = y_train_lgbm.copy()
X_val_xgb = X_val_lgbm.copy()
y_val_xgb = y_val_lgbm.copy()

print("🤖 Training XGBoost model on training split...")
model_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50, # Set in constructor
    eval_metric='rmse'        # Set in constructor
)

# Train the model
model_xgb.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False
)

# Evaluate performance
val_preds_xgb = np.clip(model_xgb.predict(X_val_xgb), 0, None)
val_rmse_xgb = root_mean_squared_error(y_val_xgb, val_preds_xgb)
val_mae_xgb = mean_absolute_error(y_val_xgb, val_preds_xgb)
val_r2_xgb = r2_score(y_val_xgb, val_preds_xgb)

print("\n📊 === XGBOOST VALIDATION PERFORMANCE ===")
print(f"Validation RMSE: {val_rmse_xgb:.5f}")
print(f"Validation MAE:  {val_mae_xgb:.5f}")
print(f"Validation R2:   {val_r2_xgb:.5f}")
print("========================================\n")

🤖 Training XGBoost model on training split...

📊 === XGBOOST VALIDATION PERFORMANCE ===
Validation RMSE: 0.02900
Validation MAE:  0.01925
Validation R2:   0.95932



In [ ]:
# --- RE-IMPORT NECESSARY LIBRARIES AND METRICS (for self-containment) ---
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from lightgbm.callback import early_stopping

print("🤖 Re-training models and generating ensemble predictions due to NameError...")

# --- RE-DEFINE FEATURES AND DATA SPLITS FOR MODELS (for self-containment) ---
# These variables (train_split, val_split, X_test, y_val) are assumed to be defined by prior cells
# If this cell still fails, please ensure all prior data preprocessing and splitting cells are executed.

lgbm_features = [
    'minutes',
    'NumberofLanes',
    'Temperature',
    'geo_demand_mean',
    'geo_demand_median',
    'geo_demand_std',
    'RoadType_enc',
    'LargeVehicles_enc',
    'Landmarks_enc',
    'Weather_enc',
    'hour_sin',
    'hour_cos',
    'day_mod7',
    'day_sin',
    'day_cos',
    'time_in_minutes',
    'geo_hour_mean',
    'geo_prefix3_enc',
    'geo_prefix4_enc',
    'geo_prefix5_enc'
]
lgbm_categorical_features = [
    'RoadType_enc',
    'LargeVehicles_enc',
    'Landmarks_enc',
    'Weather_enc',
    'day_mod7',
    'geo_prefix3_enc',
    'geo_prefix4_enc',
    'geo_prefix5_enc'
]

X_train_lgbm = train_split[lgbm_features]
y_train_lgbm = train_split['demand']
X_val_lgbm = val_split[lgbm_features]
y_val_lgbm = val_split['demand']

X_train_xgb = X_train_lgbm.copy()
y_train_xgb = y_train_lgbm.copy()
X_val_xgb = X_val_lgbm.copy()
y_val_xgb = y_val_lgbm.copy()

# --- RE-TRAIN LIGHTGBM MODEL ---
print("🤖 Re-training LightGBM model...")
model_lgbm = LGBMRegressor(
    objective='regression_l1',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1
)
model_lgbm.fit(X_train_lgbm, y_train_lgbm,
            eval_set=[(X_val_lgbm, y_val_lgbm)],
            eval_metric='mae',
            callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
            categorical_feature=lgbm_categorical_features
)

# --- RE-TRAIN XGBOOST MODEL ---
print("🤖 Re-training XGBoost model...")
model_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='rmse'
)
model_xgb.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False
)

# --- ENSEMBLE PREDICTIONS ON VALIDATION SET ---
val_preds_lgbm_ensemble = model_lgbm.predict(X_val_lgbm)
val_preds_xgb_ensemble = model_xgb.predict(X_val_xgb)
val_ensemble_preds = (val_preds_lgbm_ensemble + val_preds_xgb_ensemble) / 2
val_ensemble_preds = np.clip(val_ensemble_preds, 0, None)

# --- EVALUATE ENSEMBLE PERFORMANCE ON VALIDATION SET ---
ensemble_rmse = root_mean_squared_error(y_val, val_ensemble_preds)
ensemble_mae = mean_absolute_error(y_val, val_ensemble_preds)
ensemble_r2 = r2_score(y_val, val_ensemble_preds)

print("\n📊 === ENSEMBLE VALIDATION PERFORMANCE (LightGBM + XGBoost) ===")
print(f"Validation RMSE: {ensemble_rmse:.5f}")
print(f"Validation MAE:  {ensemble_mae:.5f}")
print(f"Validation R2:   {ensemble_r2:.5f}")
print("===========================================================\n")

# --- ENSEMBLE PREDICTIONS ON TEST SET ---
# Explicitly add the 'minutes' column to X_test if it's not already there.
# The `timestamp_to_minutes` function is assumed to be defined in a prior cell.
if 'minutes' not in X_test.columns:
    X_test['minutes'] = X_test['timestamp'].str.split(':', expand=True).astype(int).pipe(lambda x: x[0] * 60 + x[1])

X_test_lgbm_ensemble = X_test[lgbm_features]
X_test_xgb_ensemble = X_test[lgbm_features]

test_preds_lgbm = model_lgbm.predict(X_test_lgbm_ensemble)
test_preds_xgb = model_xgb.predict(X_test_xgb_ensemble)
test_ensemble_preds = (test_preds_lgbm + test_preds_xgb) / 2
test_ensemble_preds = np.clip(test_ensemble_preds, 0, None)

# --- CREATE SUBMISSION CSV ---
submission_df = pd.DataFrame({'Index': X_test['Index'], 'demand': test_ensemble_preds})
submission_df.to_csv('submission.csv', index=False)

print("✅ Submission file 'submission.csv' created successfully!")

🤖 Re-training models and generating ensemble predictions due to NameError...
🤖 Re-training LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007038 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1601
[LightGBM] [Info] Number of data points in the train set: 74602, number of used features: 19
[LightGBM] [Info] Start training from score 0.047439
🤖 Re-training XGBoost model...

📊 === ENSEMBLE VALIDATION PERFORMANCE (LightGBM + XGBoost) ===
Validation RMSE: 0.02895
Validation MAE:  0.01900
Validation R2:   0.95945

✅ Submission file 'submission.csv' created successfully!
